In [1]:
import os

from dotenv import load_dotenv
_ = load_dotenv()

In [2]:
profile = {
    "name": "John Doe",
    "age": 30,
    "city": "New York"
}

In [3]:
prompt_instruction = {
    "triage_rules":{
        "ignore": "Marketing emails",
        "notify": "Team member out sick",
        "escalate": "Legal requests"
    },
    "agent_instructions": """
    You are a helpful assistant that can answer questions and help with tasks.
    """
}




In [4]:
email = {
    "from": "support@example.com",
    "to": "john.doe@example.com",
    "subject": "Help with my account",
    "body": "I'm having trouble logging in to my account. Can you help me?"
}

In [5]:
from pydantic import BaseModel, Field
from typing_extensions import Annotated, TypedDict, Literal
from langchain_classic.chat_models import init_chat_model

In [8]:
llm = init_chat_model(
    model="LongCat-Flash-Chat",
    model_provider="openai",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.longcat.chat/openai"
)

In [7]:
class Router(BaseModel):
    """Analyze the unread email and classify it into one of the following categories: ignore, notify, or escalate"""
    reason: str = Field(
        description="Step by step reasoning for the classification"
        )
    classification: Literal["ignore", "notify", "escalate"] = Field(
        description="The classification of an email: \
        'ignore' for marketing emails, \
        'notify' for team member out sick, \
        or 'escalate' for legal requests"
        )


In [9]:
router = llm.with_structured_output(Router)

In [10]:
from prompts import triage_system_prompt, triage_user_prompt

In [11]:
system_prompt = triage_system_prompt.format(
    full_name=profile["name"],
    email_from=email["from"],
    examples = None,
    email_to=email["to"],
    subject=email["subject"],
    body=email["body"]
)

